# Stroke Detection using MediaPipe Face Mesh
Detects facial asymmetry as an early sign of stroke (FAST method).

## Step 1: Check Environment

In [ ]:
import sys
print('Python:', sys.executable)
print('Version:', sys.version)

## Step 2: Import Libraries

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
import time, urllib.request, os
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

print('cv2:', cv2.__version__)
print('mediapipe:', mp.__version__)
print('numpy:', np.__version__)

## Step 3: Initialize Face Landmarker
Downloads the model (~29MB) on first run.

In [ ]:
model_path = 'face_landmarker.task'
if not os.path.exists(model_path):
    print('Downloading model...')
    url = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
    urllib.request.urlretrieve(url, model_path)
    print('Download complete.')
else:
    print('Model found.')

options = mp_vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=model_path),
    running_mode=mp_vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5
)
detector = mp_vision.FaceLandmarker.create_from_options(options)
print('Face Landmarker ready.')

## Step 4: Landmark Indices and Thresholds

In [ ]:
MOUTH_LEFT       = 61
MOUTH_RIGHT      = 291
LEFT_EYE_TOP     = 159
LEFT_EYE_BOTTOM  = 145
RIGHT_EYE_TOP    = 386
RIGHT_EYE_BOTTOM = 374
LEFT_EYEBROW     = 105
RIGHT_EYEBROW    = 334
NOSE_TIP         = 1

BASELINE_SECONDS = 3
STROKE_THRESHOLD = 30
print('Landmarks and thresholds set.')

## Step 5: Asymmetry Functions

In [ ]:
def get_pt(lms, idx, w, h):
    return np.array([lms[idx].x * w, lms[idx].y * h])

def mouth_asym(lms, w, h):
    L, R = get_pt(lms, MOUTH_LEFT, w, h), get_pt(lms, MOUTH_RIGHT, w, h)
    width = abs(R[0] - L[0])
    return abs(L[1] - R[1]) / width if width else 0

def eye_asym(lms, w, h):
    lo = abs(get_pt(lms, LEFT_EYE_TOP, w, h)[1]  - get_pt(lms, LEFT_EYE_BOTTOM, w, h)[1])
    ro = abs(get_pt(lms, RIGHT_EYE_TOP, w, h)[1] - get_pt(lms, RIGHT_EYE_BOTTOM, w, h)[1])
    avg = (lo + ro) / 2
    return abs(lo - ro) / avg if avg else 0

def brow_asym(lms, w, h):
    nose = get_pt(lms, NOSE_TIP, w, h)
    ld = abs(nose[1] - get_pt(lms, LEFT_EYEBROW, w, h)[1])
    rd = abs(nose[1] - get_pt(lms, RIGHT_EYEBROW, w, h)[1])
    avg = (ld + rd) / 2
    return abs(ld - rd) / avg if avg else 0

# Kothari 1999: facial droop = most sensitive CPSS sign → 50%; eye signs (CN VII palsy) → 40%; brow not in CPSS → 10%
def risk_score(m, e, b):
    return min((m * 0.5 + e * 0.4 + b * 0.1) * 100, 100)

def draw_points(frame, lms, w, h):
    for idx in [MOUTH_LEFT, MOUTH_RIGHT, LEFT_EYE_TOP, LEFT_EYE_BOTTOM,
                RIGHT_EYE_TOP, RIGHT_EYE_BOTTOM, LEFT_EYEBROW, RIGHT_EYEBROW, NOSE_TIP]:
        pt = get_pt(lms, idx, w, h).astype(int)
        cv2.circle(frame, tuple(pt), 4, (0, 255, 0), -1)

print('Functions ready.')

## Step 6: Feature Extraction from Dataset Images
Reads stroke and normal face images from Kaggle datasets. Extracts mouth, eye, and brow asymmetry features using MediaPipe IMAGE mode.

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc

# Separate IMAGE mode detector (Step 3 uses VIDEO mode for live webcam)
img_options = mp_vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=model_path),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
)
img_detector = mp_vision.FaceLandmarker.create_from_options(img_options)

def extract_features_from_image(path):
    img = cv2.imread(path)
    if img is None:
        return None
    h, w = img.shape[:2]
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    try:
        result = img_detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        if not result.face_landmarks:
            return None
        lms = result.face_landmarks[0]
        m = mouth_asym(lms, w, h)
        e = eye_asym(lms, w, h)
        b = brow_asym(lms, w, h)
        return [m, e, b, risk_score(m, e, b)]
    except Exception:
        return None

def load_folder(folder, label, limit=400):
    exts = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
    files = sorted([f for f in os.listdir(folder) if f.endswith(exts)])[:limit]
    feats, lbls = [], []
    for fname in files:
        feat = extract_features_from_image(os.path.join(folder, fname))
        if feat:
            feats.append(feat)
            lbls.append(label)
    tag = 'stroke' if label else 'normal'
    print(f'{os.path.basename(folder)} ({tag}): {len(feats)}/{len(files)} faces detected')
    return feats, lbls

features, labels = [], []
for folder, label, limit in [
    ('dataset_stroke_annotated/Annotated stroke and non stroke Dataset/Stroke',    1, 400),
    ('dataset_facial_droop/Strokefaces/droopy',                                    1, 400),
    ('dataset_stroke_annotated/Annotated stroke and non stroke Dataset/NonStroke', 0, 800),
]:
    f, l = load_folder(folder, label, limit)
    features.extend(f)
    labels.extend(l)

X = np.array(features)
y = np.array(labels)
print()
print(f'Total: {len(X)} samples | Stroke: {int(sum(y))} | Normal: {int(len(y) - sum(y))}')

## Step 7: Train & Compare ML Models
Trains three classifiers and compares accuracy. Best model is auto-selected.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'SVM':                 SVC(kernel='rbf', probability=True, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_s))
    results[name] = (model, acc)
    print(f'{name}: {acc * 100:.1f}%')

plt.figure(figsize=(8, 4))
names = list(results.keys())
accs  = [results[n][1] * 100 for n in names]
bars  = plt.bar(names, accs, color=['steelblue', 'orange', 'green'])
plt.ylim(0, 110)
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy Comparison')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

best_name  = max(results, key=lambda k: results[k][1])
best_model = results[best_name][0]
print()
print(f'Best model: {best_name} ({results[best_name][1] * 100:.1f}%)')

## Step 8: Model Evaluation
Confusion matrix and ROC curve for the best model.

In [ ]:
y_pred = best_model.predict(X_test_s)
y_prob = best_model.predict_proba(X_test_s)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred)
axes[0].imshow(cm, cmap='Blues')
axes[0].set_title(f'Confusion Matrix — {best_name}')
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['Normal', 'Stroke'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['Normal', 'Stroke'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=14,
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc     = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()
print(classification_report(y_test, y_pred, target_names=['Normal', 'Stroke']))

## Step 9: Save Best Model
Saves trained model and scaler to disk for use in live detection.

In [ ]:
joblib.dump(best_model, 'stroke_model.pkl')
joblib.dump(scaler,     'stroke_scaler.pkl')
print('Saved: stroke_model.pkl')
print('Saved: stroke_scaler.pkl')

## Step 9b: Voice Guidance Setup
Installs and initializes `pyttsx3` for offline text-to-speech. Run once before the guided test.

In [ ]:
import subprocess, sys, threading

try:
    import pyttsx3
    print('pyttsx3 already installed.')
except ImportError:
    print('Installing pyttsx3...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyttsx3'])
    import pyttsx3
    print('pyttsx3 installed.')

def speak(text):
    """Fire-and-forget TTS — runs in background thread so camera loop never blocks."""
    def _run():
        try:
            engine = pyttsx3.init()
            engine.setProperty('rate', 145)
            engine.say(text)
            engine.runAndWait()
        except Exception:
            pass
    threading.Thread(target=_run, daemon=True).start()

speak('Voice guidance ready.')
print('Voice guidance ready.')

## Step 10: Guided Stroke Test — Action-Triggered Movement Asymmetry

Five-phase guided test. Each action phase starts counting **only when the action is detected** — not when the instruction appears.

| Phase | Trigger | Duration |
|-------|---------|----------|
| REST | Automatic | 4 s |
| SMILE | Smile detected (mouth corners rise) | 3 s |
| CLOSE eyes | Eyes close detected | 3 s |
| OPEN eyes wide | Wide-open detected | 3 s |
| SMILE again | Smile detected | 3 s |

Both smile attempts are averaged for reliability. Press `q` to quit.

In [ ]:
# Detection thresholds
SMILE_TRIGGER_PX = 5
CLOSE_RATIO      = 0.35
OPEN_RATIO       = 1.25
ACTION_DURATION  = 3.0
REST_DURATION    = 4.0

PHASE_REST, PHASE_SMILE1, PHASE_EYE_CLOSE, PHASE_EYE_OPEN, PHASE_SMILE2 = 0, 1, 2, 3, 4

phase_data = {
    i: {'mouth_l_y': [], 'mouth_r_y': [], 'eye_l': [], 'eye_r': []}
    for i in range(5)
}

STATE_INFO = {
    'rest':        ('REST  — keep face normal',   (255, 200,   0), True),
    'smile1_wait': ('SMILE now  :)',              (  0, 200, 255), False),
    'smile1_act':  ('SMILE  — hold it!',          (  0, 200, 255), True),
    'close_wait':  ('CLOSE your eyes',            (200, 255,   0), False),
    'close_act':   ('CLOSE  — hold it!',          (200, 255,   0), True),
    'open_wait':   ('OPEN eyes wide',             (  0, 255, 200), False),
    'open_act':    ('OPEN  — hold it!',           (  0, 255, 200), True),
    'smile2_wait': ('SMILE again  :)',            (  0, 200, 255), False),
    'smile2_act':  ('SMILE again  — hold it!',    (  0, 200, 255), True),
}

COLLECT_STATES = {
    'rest':        PHASE_REST,
    'smile1_act':  PHASE_SMILE1,
    'close_act':   PHASE_EYE_CLOSE,
    'open_act':    PHASE_EYE_OPEN,
    'smile2_act':  PHASE_SMILE2,
}

VOICE_MESSAGES = {
    'rest':        'Keep your face normal.',
    'smile1_wait': 'Please smile now.',
    'smile1_act':  'Good. Hold your smile.',
    'close_wait':  'Now close your eyes.',
    'close_act':   'Good. Keep them closed.',
    'open_wait':   'Now open your eyes wide.',
    'open_act':    'Good. Keep them open.',
    'smile2_wait': 'Smile again please.',
    'smile2_act':  'Good. Hold your smile.',
}

def get_measurements(lms, w, h):
    ml_y = get_pt(lms, MOUTH_LEFT,         w, h)[1]
    mr_y = get_pt(lms, MOUTH_RIGHT,        w, h)[1]
    el   = abs(get_pt(lms, LEFT_EYE_TOP,   w, h)[1] - get_pt(lms, LEFT_EYE_BOTTOM,  w, h)[1])
    er   = abs(get_pt(lms, RIGHT_EYE_TOP,  w, h)[1] - get_pt(lms, RIGHT_EYE_BOTTOM, w, h)[1])
    return ml_y, mr_y, el, er

def phase_mean(idx):
    d = phase_data[idx]
    if not d['mouth_l_y']:
        return 0.0, 0.0, 0.0, 0.0
    return (np.mean(d['mouth_l_y']), np.mean(d['mouth_r_y']),
            np.mean(d['eye_l']),     np.mean(d['eye_r']))

def compute_guided_result():
    r_ml, r_mr, r_el, r_er = phase_mean(PHASE_REST)

    # Smile: peak upward movement per side — same as webapp (uses max, not mean)
    def smile_peak(phase_idx):
        d = phase_data[phase_idx]
        if not d['mouth_l_y']:
            return 0.0, 0.0, 0.0
        L = max(r_ml - y for y in d['mouth_l_y'])
        R = max(r_mr - y for y in d['mouth_r_y'])
        denom = max(L, R, 1e-4)
        return abs(L - R) / denom, L, R

    asym1, ld1, rd1 = smile_peak(PHASE_SMILE1)
    asym2, ld2, rd2 = smile_peak(PHASE_SMILE2)
    smile_asym = (asym1 + asym2) / 2
    ld = (ld1 + ld2) / 2
    rd = (rd1 + rd2) / 2

    # Close: minimum eye height (most closed frame) — same as webapp
    d_c = phase_data[PHASE_EYE_CLOSE]
    le_min = min(d_c['eye_l']) if d_c['eye_l'] else r_el
    re_min = min(d_c['eye_r']) if d_c['eye_r'] else r_er
    lClose = 1 - le_min / (r_el + 1e-6)
    rClose = 1 - re_min / (r_er + 1e-6)
    close_asym = abs(lClose - rClose) / max(lClose, rClose, 0.01)

    # Open: maximum eye height (most open frame) as ratio above rest — same as webapp
    d_o = phase_data[PHASE_EYE_OPEN]
    le_max = max(d_o['eye_l']) if d_o['eye_l'] else r_el
    re_max = max(d_o['eye_r']) if d_o['eye_r'] else r_er
    lOpen = le_max / (r_el + 1e-6) - 1
    rOpen = re_max / (r_er + 1e-6) - 1
    open_asym = abs(lOpen - rOpen) / max(lOpen, rOpen, 0.01)

    score = min((smile_asym * 0.5 + close_asym * 0.25 + open_asym * 0.25) * 100, 100)
    return score, smile_asym, asym1, asym2, close_asym, open_asym, ld, rd

rest_vals = None

def is_smiling(ml_y, mr_y):
    r_ml, r_mr, _, _ = rest_vals
    return ((r_ml - ml_y) + (r_mr - mr_y)) / 2 > SMILE_TRIGGER_PX

def is_eyes_closed(el, er):
    _, _, r_el, r_er = rest_vals
    return (el + er) / 2 < (r_el + r_er) / 2 * CLOSE_RATIO

def is_eyes_open_wide(el, er):
    _, _, r_el, r_er = rest_vals
    return (el + er) / 2 > (r_el + r_er) / 2 * OPEN_RATIO

def transition(new_state, t):
    global state, state_start
    state, state_start = new_state, t
    if new_state in VOICE_MESSAGES:
        speak(VOICE_MESSAGES[new_state])

# --- Main loop ---
cap           = cv2.VideoCapture(0)
ts_g          = 0
state         = 'rest'
state_start   = time.time()
guided_result = None

speak(VOICE_MESSAGES['rest'])
print('Guided stroke test started. Press q to quit.')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    h, w    = frame.shape[:2]
    now     = time.time()
    elapsed = now - state_start
    ts_g   += 1

    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = detector.detect_for_video(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts_g
    )

    ml_y = mr_y = el = er = None
    if result.face_landmarks:
        lms = result.face_landmarks[0]
        ml_y, mr_y, el, er = get_measurements(lms, w, h)
        draw_points(frame, lms, w, h)
        if state in COLLECT_STATES:
            d = phase_data[COLLECT_STATES[state]]
            d['mouth_l_y'].append(ml_y)
            d['mouth_r_y'].append(mr_y)
            d['eye_l'].append(el)
            d['eye_r'].append(er)

    # --- State machine ---
    if state == 'rest':
        if elapsed >= REST_DURATION:
            rest_vals = phase_mean(PHASE_REST)
            transition('smile1_wait', now)

    elif state == 'smile1_wait':
        if ml_y is not None and is_smiling(ml_y, mr_y):
            transition('smile1_act', now)

    elif state == 'smile1_act':
        if elapsed >= ACTION_DURATION:
            transition('close_wait', now)

    elif state == 'close_wait':
        if el is not None and is_eyes_closed(el, er):
            transition('close_act', now)

    elif state == 'close_act':
        if elapsed >= ACTION_DURATION:
            transition('open_wait', now)

    elif state == 'open_wait':
        if el is not None and is_eyes_open_wide(el, er):
            transition('open_act', now)

    elif state == 'open_act':
        if elapsed >= ACTION_DURATION:
            transition('smile2_wait', now)

    elif state == 'smile2_wait':
        if ml_y is not None and is_smiling(ml_y, mr_y):
            transition('smile2_act', now)

    elif state == 'smile2_act':
        if elapsed >= ACTION_DURATION:
            state = 'done'
            guided_result = compute_guided_result()
            score, smile_a, a1, a2, close_a, open_a, ld, rd = guided_result
            if score > 60:
                speak('High risk detected. Please consult a doctor immediately.')
                verdict = 'HIGH RISK — see a doctor immediately'
            elif score > 30:
                speak('Moderate risk detected. Please monitor closely.')
                verdict = 'MODERATE — monitor closely'
            else:
                speak('Test complete. Result is normal.')
                verdict = 'NORMAL'
            print(f'\n=== GUIDED TEST RESULT ===')
            print(f'Final Score        : {score:.1f} / 100')
            print(f'Smile asymmetry    : {smile_a:.3f}  (attempt1={a1:.3f}  attempt2={a2:.3f})')
            print(f'Eye close asymmetry: {close_a:.3f}')
            print(f'Eye open asymmetry : {open_a:.3f}')
            print(f'Verdict            : {verdict}')

    # --- HUD ---
    cv2.rectangle(frame, (0, 0), (w, 60), (0, 0, 0), -1)

    if state == 'done' and guided_result:
        score = guided_result[0]
        if score > 60:
            txt, vc = f'HIGH RISK    Score: {score:.0f}/100', (0,   0, 255)
        elif score > 30:
            txt, vc = f'MODERATE     Score: {score:.0f}/100', (0, 165, 255)
        else:
            txt, vc = f'NORMAL       Score: {score:.0f}/100', (0, 200,   0)
        cv2.putText(frame, txt, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, vc, 2)

    elif state in STATE_INFO:
        label, color, is_timed = STATE_INFO[state]
        if is_timed:
            duration  = REST_DURATION if state == 'rest' else ACTION_DURATION
            remaining = max(0, duration - elapsed)
            display   = f'{label}  ({int(remaining) + 1}s)'
            prog      = int((elapsed / duration) * w)
            cv2.rectangle(frame, (0, 55), (prog, 60), color, -1)
        else:
            dot     = '...' if int(now * 2) % 2 == 0 else '   '
            display = f'{label}  {dot}'
        cv2.putText(frame, display, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    else:
        cv2.putText(frame, 'No face detected', (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    cv2.imshow('Guided Stroke Test', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print('Test complete. Run Step 11 to visualize results.')

## Step 11: Visualize Guided Test Results
Bar chart showing movement asymmetry per phase, plus smile corner comparison (left vs right).

In [ ]:
if guided_result is None:
    print('No guided test result found. Run Step 10 first.')
else:
    score, smile_a, a1, a2, close_a, open_a, ld, rd = guided_result

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: asymmetry per phase
    labels = ['Smile 1', 'Smile 2', 'Smile\n(avg)', 'Eye Close', 'Eye Open']
    values = [a1*100, a2*100, smile_a*100, close_a*100, open_a*100]
    bar_colors = ['red' if v > 60 else 'orange' if v > 30 else 'steelblue' for v in values]
    bar_colors[2] = 'purple'   # avg bar stands out
    axes[0].bar(labels, values, color=bar_colors)
    axes[0].axhline(60, color='red',    linestyle='--', linewidth=1.5, label='High Risk (>60)')
    axes[0].axhline(30, color='orange', linestyle='--', linewidth=1.5, label='Moderate (>30)')
    axes[0].set_ylim(0, 110)
    axes[0].set_ylabel('Asymmetry Score')
    axes[0].set_title('Movement Asymmetry by Phase')
    axes[0].legend(fontsize=8)
    for i, v in enumerate(values):
        axes[0].text(i, v + 1.5, f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')

    # Right: smile corner comparison (averaged)
    axes[1].bar(['Left Corner', 'Right Corner'], [ld, rd], color=['steelblue', 'tomato'])
    axes[1].set_ylabel('Upward Movement (pixels)')
    axes[1].set_title('Smile Corner Movement — Left vs Right\n(avg of both attempts)')
    axes[1].axhline(0, color='black', linewidth=0.8)
    for i, v in enumerate([ld, rd]):
        axes[1].text(i, max(v, 0) + 0.3, f'{v:.1f}px', ha='center', fontweight='bold')

    verdict = ('HIGH RISK' if score > 60 else 'MODERATE' if score > 30 else 'NORMAL')
    plt.suptitle(f'Guided Test — Final Score: {score:.1f}/100  |  Verdict: {verdict}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'Score: {score:.1f}/100  |  Verdict: {verdict}')
    print(f'  Smile attempt 1   : {a1:.3f}')
    print(f'  Smile attempt 2   : {a2:.3f}')
    print(f'  Smile avg         : {smile_a:.3f}  (L={ld:.1f}px  R={rd:.1f}px)')
    print(f'  Eye close         : {close_a:.3f}')
    print(f'  Eye open          : {open_a:.3f}')